In [ ]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
import matplotlib.pyplot as plt
import datetime
import os
import glob
import time
from utils.model import build_enhanced_discriminator,build_unet_lstm_generator
from utils.train import train_step
from utils.metrics import integrate_rpeak_into_training
from utils.strategy import select_final_model,freeze_for_personalization

# Set seeds
tf.random.set_seed(42)
np.random.seed(42)


class SmartCheckpointManager:
    def __init__(self, generator, save_dir, subject_id="01", patience=20):
        self.generator = generator
        self.save_dir = save_dir
        self.subject_id = subject_id
        self.patience = patience
        
        self.best_pcc = -1.0
        self.best_rpa = -1.0
        self.best_score = -1.0
        
        self.best_metrics = {
            'best_pcc_model': {'pcc': -1.0, 'rpa': -1.0, 'mse': -1.0, 'epoch': 0},
            'best_rpa_model': {'pcc': -1.0, 'rpa': -1.0, 'mse': -1.0, 'epoch': 0},
            'best_overall_model': {'pcc': -1.0, 'rpa': -1.0, 'mse': -1.0, 'epoch': 0}
        }

        self.wait = 0
        self.threshold_met = False
        os.makedirs(save_dir, exist_ok=True)

    def calculate_score(self, pcc, rpa):
        if rpa < 0.85:
            return 0.2 * pcc + 0.8 * rpa
        else:
            return 0.6 * pcc + 0.4 * rpa

    def check_and_save(self, epoch, pcc, rpa, mse):
        stop_training = False
        score = self.calculate_score(pcc, rpa)
        current_epoch = epoch + 1  
        
        # 1.  Best PCC
        if pcc > self.best_pcc:
            self.best_pcc = pcc
            self.best_metrics['best_pcc_model'] = {
                'pcc': pcc, 
                'rpa': rpa, 
                'mse': mse, 
                'epoch': current_epoch
            } 
            self.generator.save(os.path.join(self.save_dir, f'{self.subject_id}_best_pcc_model.h5'))
            
        # 2.  Best RPA
        if rpa > self.best_rpa:
            self.best_rpa = rpa
            self.best_metrics['best_rpa_model'] = {
                'pcc': pcc, 
                'rpa': rpa, 
                'mse': mse, 
                'epoch': current_epoch
            }
            self.generator.save(os.path.join(self.save_dir, f'{self.subject_id}_best_rpa_model.h5'))

        # 3.  Best Overall
        if score > self.best_score:
            self.best_score = score
            self.best_metrics['best_overall_model'] = {
                'pcc': pcc, 
                'rpa': rpa, 
                'mse': mse, 
                'epoch': current_epoch
            }
            self.wait = 0 
            self.generator.save(os.path.join(self.save_dir, f'{self.subject_id}_best_overall_model.h5'))
            print(f"    ★ Saved Best Overall (Epoch {current_epoch} | Score: {score:.4f} | PCC: {pcc:.4f} | RPA: {rpa:.4f})")
        else:
            if self.threshold_met:
                self.wait += 1

        # 4. early stopping logic
        if pcc > 0.85 and rpa > 0.90:
            if not self.threshold_met:
                print("    >>> Thresholds Met! Entering fine-tuning patience phase...")
                self.threshold_met = True
                self.wait = 0
        
        if self.threshold_met and self.wait >= self.patience:
            print(f"\nSTOPPING: Metrics stable for {self.patience} epochs.")
            stop_training = True
            
        return stop_training



def train(dataset, test_dataset, epochs, generator, discriminator):
    
    save_path = '/root/tmp/PulseGUARD/model'
    log_dir = os.path.join("logs", "personalization", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    summary_writer = tf.summary.create_file_writer(log_dir)

    checkpoint_manager = SmartCheckpointManager(
        generator=generator, 
        save_dir=save_path, 
        subject_id="21", 
        patience=20
    )

    for epoch in range(epochs):
        print(f"Epoch {epoch+1}/{epochs}")
    
        # --- training ---
        for ppg_batch, ecg_batch in dataset:
            g_loss, d_loss = train_step(ppg_batch, ecg_batch, generator, discriminator)
      
        print(f"  - Losses: G={g_loss:.4f}, D={d_loss:.4f}")    
        mse_list, pcc_list, rpa_list = [], [], []
        for test_ppg, test_ecg in test_dataset:
            pred_ecg = generator(test_ppg, training=False).numpy()
            m = integrate_rpeak_into_training(test_ecg, pred_ecg)
            mse_list.append(m['MSE'])
            pcc_list.append(m['PCC'])
            rpa_list.append(m['R_peak_accuracy'])
            
        epoch_mse = np.mean(mse_list)
        epoch_pcc = np.mean(pcc_list)
        epoch_rpa = np.mean(rpa_list)

        print(f"  [Val] PCC: {epoch_pcc:.4f} | RPA: {epoch_rpa:.4f} | MSE: {epoch_mse:.4f}")

        # --- Logging ---
        with summary_writer.as_default():
            tf.summary.scalar('Generator Loss', g_loss, step=epoch)
            tf.summary.scalar('MSE', epoch_mse, step=epoch)
            tf.summary.scalar('PCC', epoch_pcc, step=epoch)
            tf.summary.scalar('R-peak Accuracy', epoch_rpa, step=epoch)
            # 记录 Score
            score = checkpoint_manager.calculate_score(epoch_pcc, epoch_rpa)
            tf.summary.scalar('Composite Score', score, step=epoch)

        # --- Storage and early stop check ---
        should_stop = checkpoint_manager.check_and_save(epoch, epoch_pcc, epoch_rpa, epoch_mse)
        
        if should_stop:
            break

    print("\n" + "="*60)
    print("TRAINING FINISHED. STARTING MODEL SELECTION.")
    print("="*60)
    print("Candidate Models Overview:")
    print(f"{'Model Type':<20} | {'Epoch':<6} | {'PCC':<8} | {'RPA':<8} | {'MSE':<10}")
    print("-" * 65)
    
    for k, v in checkpoint_manager.best_metrics.items():
        ep = v['epoch'] if v['epoch'] > 0 else "-"
        pcc = f"{v['pcc']:.4f}" if v['pcc'] != -1 else "-"
        rpa = f"{v['rpa']:.4f}" if v['rpa'] != -1 else "-"
        mse = f"{v['mse']:.4f}" if v['mse'] != -1 else "-"
        
        name_map = {
            'best_pcc_model': 'Best PCC',
            'best_rpa_model': 'Best RPA',
            'best_overall_model': 'Best Overall'
        }
        name = name_map.get(k, k)
        
        print(f"{name:<20} | {ep:<6} | {pcc:<8} | {rpa:<8} | {mse:<10}")
    print("-" * 65)
    
    # 2. Implement the selection strategy
    selected_model_key = select_final_model(checkpoint_manager.best_metrics, safety_threshold=0.85)
    
    # 3. Load the selected model weights
    final_model_path = os.path.join(save_path, f"21_{selected_model_key}.h5")
    
    if os.path.exists(final_model_path):
        print(f"\nLOADING FINAL WEIGHTS: {final_model_path}")
        generator.load_weights(final_model_path)
        
        final_save_path = os.path.join(save_path, '21_final_selected_generator.h5')
        generator.save(final_save_path)
        print(f"✓ Final selected model saved to: {final_save_path}")
    else:
        print(f"❌ Error: Selected model file {final_model_path} not found!")

    return generator



In [ ]:
# Initialize models
seq_len = 375
generator = build_unet_lstm_generator(seq_len)
generator = freeze_for_personalization(generator)
discriminator = build_enhanced_discriminator(seq_len)

# Dummy run to set inputs
generator(tf.keras.Input(shape=(seq_len, 1), dtype=tf.float32))
discriminator(tf.keras.Input(shape=(seq_len, 1), dtype=tf.float32))

# Load pre-trained weights
try:
    generator.load_weights('/root/tmp/PulseGUARD/model/best_generator_mse.h5')
    print("✓ Loaded best generator weights.")
except Exception as e:
    print(f"! Warning: Could not load weights: {e}")

# Freeze Discriminator
discriminator.trainable = False
print("Discriminator weights frozen")

# Data Loading & Preprocessing
def preprocess_data(X, y):
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    X = 2 * (X - np.min(X)) / (np.max(X) - np.min(X)) - 1
    y = 2 * (y - np.min(y)) / (np.max(y) - np.min(y)) - 1
    return X, y

# Load Data (Please ensure these paths exist)
try:
    X_train = np.load('/root/tmp/PulseGUARD/01_pulse_train.npy')
    y_train = np.load('/root/tmp/PulseGUARD/01_ecg_train.npy')
    X_test = np.load('/root/tmp/PulseGUARD/01_pulse_test.npy')
    y_test = np.load('/root/tmp/PulseGUARD/01_ecg_test.npy')

    X_train, y_train = preprocess_data(X_train, y_train)
    X_test, y_test = preprocess_data(X_test, y_test)

    print(f"Train shapes: {X_train.shape}, {y_train.shape}")
    print(f"Test shapes: {X_test.shape}, {y_test.shape}")

    train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1000).batch(8)
    test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(8)

    # Start Training
    generator = train(train_dataset, test_dataset, epochs=200, generator=generator, discriminator=discriminator) # Increased epochs to allow early stopping to work

except FileNotFoundError:
    print("Error: Data files not found. Please check paths.")